---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [28]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: /Users/emmas/Desktop/Ingineria AI/echochamber/echochamber-project-team-4
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [29]:
import pandas as pd
import random

corpus = pd.read_json("~/Desktop/Ingineria AI/echochamber/echochamber-project-team-4//data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[RadioClasic] Dacă o "putere mare" i-ar scoate pe ru5i, chinezi și cubanezi din Venezuela, și 
[RecorderRomania] Bravo Recorder! Foarte bun reportajul! Doar prin faza de la finalul reportajului
[@CălinGeorgescu-CanalulOficial] MS Regina Elisabeta a României a tradus sub pseudonimul Carmen Sylva poeziile lu


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [31]:
# modifica dupa preferinte

AXA_1 = "people_vs_elite"
AXA_2 = "anti_corruption"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
people_vs_elite = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
anti_corruption = comentariul folosește limbaj religios pentru a interpreta politica.

In [32]:
AXA_1_DEFINITION = """
people_vs_elite măsoară dacă textul prezintă un conflict între
„oamenii obișnuiți” și „elite” politice, economice sau instituționale.
Exemple: „elitele conduc tot”, „poporul e ignorat”, „sistemul e împotriva oamenilor”.

0 = absent (nu apare această idee)
1 = prezent (există referințe la conflictul oameni vs elite)
2 = dominant (tema principală a comentariului)
"""

AXA_2_DEFINITION = """
anti_corruption măsoară dacă textul vorbește despre corupție,
furt, abuz de putere sau politicieni corupți.
Exemple: „toți fură”, „clasa politică e coruptă”, „au furat banii țării”.

0 = absent (nu apare tema corupției)
1 = prezent (există referințe la corupție)
2 = dominant (corupția este tema centrală)
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [33]:
MINI_PROMPT = f"""
Ești un analist de discurs politic online.

SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}

CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru

{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2

DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}

REGULI:
1. Codează doar ce apare explicit în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic:
 target="none" 
 stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 
0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
8. Nu adăuga explicații suplimentare.

FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un analist de discurs politic online.

SARCINĂ:
Adnotează comentariul folosind două axe:
1. people_vs_elite
2. anti_corruption

CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru

people_vs_elite = 0 / 1 / 2
anti_corruption = 0 / 1 / 2

DEFINIȚII:

people_vs_elite măsoară dacă textul prezintă un conflict între
„oamenii obișnuiți” și „elite” politice, economice sau instituționale.
Exemple: „elitele conduc tot”, „poporul e ignorat”, „sistemul e împotriva oamenilor”.

0 = absent (nu apare această idee)
1 = prezent (există referințe la conflictul oameni vs elite)
2 = dominant (tema principală a comentariului)


anti_corruption măsoară dacă textul vorbește despre corupție,
furt, abuz de putere sau politicieni corupți.
Exemple: „toți fură”, „clasa politică e coruptă”, „au furat banii țării”.

0 = absent (nu apare

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [34]:
TESTS = corpus.sample(5, random_state=80)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
38,yt_SEVle95uBDo_UgwnnI734Efikc_bhRx4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Pacea de la București este morandumul inițiat ...
269,yt_Sj4fQKlMOro_Ugwv7I5JKIFgfBiZ3Al4AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Mulțumim pentru tot! Sper ca acest videoclip s...
223,yt_N5paaKW5FKc_Ugyj5kc74ijo6DsNBnt4AaABAg,PressOneRomania,Molia. Metamorfoza „doctorului de suflete” Cri...,Fătuțo! Cât și cine te-a plătit să împroşti cu...
311,yt_O5ZScB10LF8_UgwhgZLxRDKBdCA9agB4AaABAg,roxindaniel,Regele Ungariei pus pe fugă de Ștefan cel Mare...,CINSTE SI RESPECT DOMNULE ROXIN SI TUTUROR ERO...
60,yt_VDiv4TBODF8_Ugzj-iclKtnZDg2gZMd4AaABAg,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Sfidarea tipică lichelelor de genul lui. Dezgu...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [48]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [49]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [50]:
def llm(system, user, max_tokens=700):
    prompt = system + "\n\n" + user

    response = client_now.models.generate_content(
        model=model_now,
        contents=prompt
    )

    return response.text

In [51]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API Key not found. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API Key not found. Please pass a valid API key.'}]}}

## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?


Am ales axele people_vs_elite și anti_corruption, deoarece sunt relevante pentru analiza discursului politic online și pot surprinde atât tensiunea socială dintre cetățeni și elite, cât și temele frecvente legate de corupție.
Modelul a fost capabil să returneze un JSON valid în majoritatea cazurilor, respectând structura cerută (target, stance, tone și codarea celor două axe). Totuși, în unele cazuri au apărut probleme legate de consistența răspunsului sau erori de apel API, ceea ce a afectat rularea completă.
Cea mai mare problemă a fost instabilitatea integrării dintre clientul LLM și endpoint-ul folosit, ceea ce a dus la erori în timpul execuției și rezultate incomplete.
Dacă aș îmbunătăți promptul, aș adăuga exemple (few-shot) de adnotări corecte și aș restricționa mai strict formatul de ieșire pentru a reduce variațiile și erorile de interpretare ale modelului.